# V6 compounding stat thresholds (win rate when ALL agree)

Configure **which stats** must simultaneously pass **per-stat min edges** and **agree on the same side** (home vs away). Only those games get a compound pick; **n** is how many games qualify.

Uses the same diff conventions as `v6_stat_effectiveness_by_year` / V6 (xFIP: lower home xFIP is better; bullpen gap filled with 0 when missing).

**Outputs:** long table by season, pivots, optional sweep over one threshold.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

FINAL_STATES = {"Final", "Game Over", "Completed Early"}
PEN_ABS = 0.75

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"

# --- User config: stats to require (subset of keys below) ---
COMPOUND_STATS = [
    "sp_kbb_diff",
    "offense_diff",
]

# Per-stat min edge (same semantics as by-year notebook)
MIN_EDGE_DEFAULT = 0.0
MIN_EDGES = {
    "sp_kbb_diff": 10.0, # home - away
    "sp_xfip_diff": 1.0, # home - away
    "offense_diff": 10.0, # home - away
    "offense_platoon_diff": 10.0, # home - away
    "pen_gap_fip": 1.0, # home - away
}

# None = all parquet seasons found; else e.g. [2024, 2025]
SEASONS_FILTER = None


STAT_META = {
    "sp_kbb_diff": {"higher_home_better": True, "pen_filled": False},
    "sp_xfip_diff": {"higher_home_better": False, "pen_filled": False},
    "offense_diff": {"higher_home_better": True, "pen_filled": False},
    "offense_platoon_diff": {"higher_home_better": True, "pen_filled": False},
    "pen_gap_fip": {"higher_home_better": False, "pen_filled": True},
}


def discover_season_parquets(data_dir: Path):
    out = []
    for p in sorted(data_dir.glob("games_*.parquet")):
        try:
            year = int(p.stem.split("_")[-1])
        except ValueError:
            continue
        out.append((year, p))
    return out


pairs = discover_season_parquets(DATA)
if SEASONS_FILTER is not None:
    sset = set(SEASONS_FILTER)
    pairs = [(y, p) for y, p in pairs if y in sset]
if not pairs:
    raise FileNotFoundError(f"No parquet after filter under {DATA.resolve()}")

print("Seasons:", [y for y, _ in pairs])
print("COMPOUND_STATS:", COMPOUND_STATS)
print("MIN_EDGES used:", {k: MIN_EDGES.get(k, MIN_EDGE_DEFAULT) for k in COMPOUND_STATS})

Seasons: [2024, 2025, 2026]
COMPOUND_STATS: ['sp_kbb_diff', 'offense_diff']
MIN_EDGES used: {'sp_kbb_diff': 5.0, 'offense_diff': 5.0}


In [2]:
# Helpers: pen gap + per-row direction + compound favorite won


def preferred_pen_cols(frame: pd.DataFrame):
    if "home_pen_roll14_fip" in frame.columns:
        home_pen = frame["home_pen_roll14_fip"]
        if "home_pen_season_fip" in frame.columns:
            home_pen = home_pen.combine_first(frame["home_pen_season_fip"])
    elif "home_pen_season_fip" in frame.columns:
        home_pen = frame["home_pen_season_fip"]
    else:
        home_pen = pd.Series(np.nan, index=frame.index)

    if "away_pen_roll14_fip" in frame.columns:
        away_pen = frame["away_pen_roll14_fip"]
        if "away_pen_season_fip" in frame.columns:
            away_pen = away_pen.combine_first(frame["away_pen_season_fip"])
    elif "away_pen_season_fip" in frame.columns:
        away_pen = frame["away_pen_season_fip"]
    else:
        away_pen = pd.Series(np.nan, index=frame.index)

    return home_pen, away_pen


def stat_direction(val: float, min_edge: float, *, higher_home_better: bool) -> int:
    """+1 home favored, -1 away favored, 0 no pick / missing."""
    if val is None:
        return 0
    if isinstance(val, (float, np.floating)) and np.isnan(val):
        return 0
    m = float(min_edge)
    if np.isclose(m, 0.0):
        if higher_home_better:
            return 1 if val >= 0 else -1
        return 1 if val <= 0 else -1
    if higher_home_better:
        if val >= m:
            return 1
        if val <= -m:
            return -1
    else:
        if val <= -m:
            return 1
        if val >= m:
            return -1
    return 0


def add_pen_gap(w: pd.DataFrame) -> pd.DataFrame:
    out = w.copy()
    hp, ap = preferred_pen_cols(out)
    out["pen_gap_fip"] = hp - ap # changed to hp - ap to match sp_xfip_diff
    return out


def compound_favwon_series(w: pd.DataFrame, stats: list[str], min_edges: dict, default_m: float) -> pd.Series:
    """NaN unless all stats agree on same non-zero direction; then 1 if that side won else 0."""
    y = pd.to_numeric(w["home_win"], errors="coerce")
    dirs = []
    for col in stats:
        meta = STAT_META[col]
        if meta["pen_filled"]:
            v = pd.to_numeric(w["pen_gap_fip"], errors="coerce").fillna(0.0)
        elif col in w.columns:
            v = pd.to_numeric(w[col], errors="coerce")
        else:
            v = pd.Series(np.nan, index=w.index)

        m = float(min_edges.get(col, default_m))
        d = np.array([stat_direction(float(x) if pd.notna(x) else np.nan, m, higher_home_better=meta["higher_home_better"]) for x in v], dtype=int)
        dirs.append(d)

    mat = np.vstack(dirs)
    agree = (mat != 0).all(axis=0) & (np.max(mat, axis=0) == np.min(mat, axis=0))
    first = mat[0, :].astype(float)
    first[~agree] = np.nan
    prefer_home = first == 1
    out = np.where(np.isnan(first), np.nan, np.where(prefer_home, y == 1.0, y == 0.0))
    return pd.Series(out, index=w.index, dtype="float")


def compound_summary(w: pd.DataFrame, stats: list[str], min_edges: dict, default_m: float, season: int) -> dict:
    s = compound_favwon_series(w, stats, min_edges, default_m).dropna()
    n = int(len(s))
    hw = pd.to_numeric(w["home_win"], errors="coerce")
    baseline = float(hw.mean()) if len(hw.dropna()) else float("nan")
    if n == 0:
        return {
            "season": season,
            "compound_label": "+".join(stats),
            "n": 0,
            "favorite_win_rate": np.nan,
            "se": np.nan,
            "baseline_home_win_rate": baseline,
        }
    p = float(s.mean())
    se = float(np.sqrt(p * (1.0 - p) / n))
    return {
        "season": season,
        "compound_label": "+".join(stats),
        "n": n,
        "favorite_win_rate": p,
        "se": se,
        "baseline_home_win_rate": baseline,
    }

In [3]:
# Run all seasons

bad = [c for c in COMPOUND_STATS if c not in STAT_META]
if bad:
    raise ValueError(f"Unknown stats: {bad}. Allowed: {list(STAT_META)}")

rows = []
for season, path in pairs:
    df = pd.read_parquet(path).copy()
    df["detailed_state"] = df["detailed_state"].astype(str)
    w = df[df["detailed_state"].isin(FINAL_STATES)].copy()
    w = w[w["home_win"].notna()].copy()
    if len(w) == 0:
        continue
    w = add_pen_gap(w)
    rows.append(compound_summary(w, COMPOUND_STATS, MIN_EDGES, MIN_EDGE_DEFAULT, season))
    print(season, "n_completed", len(w), "compound_n", rows[-1]["n"])

compound_by_year = pd.DataFrame(rows).sort_values("season").reset_index(drop=True)
display(compound_by_year)

if not compound_by_year.empty:
    rp = compound_by_year.pivot(index="compound_label", columns="season", values="favorite_win_rate")
    npv = compound_by_year.pivot(index="compound_label", columns="season", values="n")
    print("Win rate pivot")
    display(rp)
    print("n pivot")
    display(npv)

2024 n_completed 2959 compound_n 459
2025 n_completed 2943 compound_n 440
2026 n_completed 619 compound_n 110


,season,compound_label,n,favorite_win_rate,se,baseline_home_win_rate
0,2024,sp_kbb_diff+offense_diff,459,0.6601,0.0221,0.5120
1,2025,sp_kbb_diff+offense_diff,440,0.6455,0.0228,0.5433
2,2026,sp_kbb_diff+offense_diff,110,0.6182,0.0463,0.5800


Win rate pivot


season,2024,2025,2026
compound_label,,,
sp_kbb_diff+offense_diff,0.6601,0.6455,0.6182


n pivot


season,2024,2025,2026
compound_label,,,
sp_kbb_diff+offense_diff,459,440,110


In [4]:
# Optional: sweep one stat's min edge (others fixed). Edit SWEEP_STAT / sweep_vals.

SWEEP_STAT = "sp_kbb_diff"
sweep_vals = [0.0, 0.5, 1.0, 1.5, 2.0]

rows_s = []
for m in sweep_vals:
    me = dict(MIN_EDGES)
    me[SWEEP_STAT] = float(m)
    for season, path in pairs:
        df = pd.read_parquet(path).copy()
        df["detailed_state"] = df["detailed_state"].astype(str)
        w = df[df["detailed_state"].isin(FINAL_STATES)].copy()
        w = w[w["home_win"].notna()].copy()
        if len(w) == 0:
            continue
        w = add_pen_gap(w)
        r = compound_summary(w, COMPOUND_STATS, me, MIN_EDGE_DEFAULT, season)
        r["sweep_stat"] = SWEEP_STAT
        r["sweep_min_edge"] = float(m)
        rows_s.append(r)

sweep_df = pd.DataFrame(rows_s)
if not sweep_df.empty:
    pv = sweep_df.pivot(index="sweep_min_edge", columns="season", values="favorite_win_rate")
    pn = sweep_df.pivot(index="sweep_min_edge", columns="season", values="n")
    print("Sweep win rate:", SWEEP_STAT)
    display(pv)
    print("Sweep n:")
    display(pn)

Sweep win rate: sp_kbb_diff


season,2024,2025,2026
sweep_min_edge,,,
0.0000,0.6368,0.6017,0.6391
0.5000,0.6381,0.6099,0.6258
1.0000,0.6402,0.6074,0.6273
1.5000,0.6416,0.6059,0.6203
2.0000,0.6491,0.6119,0.6149


Sweep n:


season,2024,2025,2026
sweep_min_edge,,,
0.0000,848,821,169
0.5000,818,787,163
1.0000,781,754,161
1.5000,745,713,158
2.0000,704,670,148
